## Dataset Information

The "spam" concept is diverse: advertisements for products/web sites, make money fast schemes, chain letters, pornography...

The SMS Spam Collection is a set of SMS tagged messages that have been collected for SMS Spam research. It contains one set of SMS messages in English of 5,574 messages, tagged according being ham (legitimate) or spam.

## Attributes

- SMS Messages
- Label (spam/ham)

In [ ]:
# --- Data Directory Setup (auto-generated) ---
from pathlib import Path
import os

# Resolve data directory relative to workspace root
def _find_data_dir():
    """Find the data directory by walking up from notebook location."""
    candidates = [
        Path.cwd().parent / "data" / "NLP Projects 31 - SMS Spam Detection Analysis",
        Path.cwd() / "data" / "NLP Projects 31 - SMS Spam Detection Analysis",
        Path(".").resolve().parent / "data" / "NLP Projects 31 - SMS Spam Detection Analysis",
    ]
    for c in candidates:
        if c.exists():
            return c
    # Fallback: current directory
    return Path(".")

DATA_DIR = _find_data_dir()
print(f"Data directory: {DATA_DIR}")

## Import modules

In [1]:
import pandas as pd
import numpy as np
import nltk
import re
from nltk.corpus import stopwords

## Loading the dataset

In [2]:
df = pd.read_csv(str(DATA_DIR / 'spam.csv'))
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [3]:
# get necessary columns for processing
df = df[['v2', 'v1']]
# df.rename(columns={'v2': 'messages', 'v1': 'label'}, inplace=True)
df = df.rename(columns={'v2': 'messages', 'v1': 'label'})
df.head()

,messages,label
0,"Go until jurong point, crazy.. Available only ...",ham
1,Ok lar... Joking wif u oni...,ham
2,Free entry in 2 a wkly comp to win FA Cup fina...,spam
3,U dun say so early hor... U c already then say...,ham
4,"Nah I don't think he goes to usf, he lives aro...",ham


## Preprocessing the dataset

In [4]:
# check for null values
df.isnull().sum()

messages    0
label       0
dtype: int64

In [5]:
STOPWORDS = set(stopwords.words('english'))

def clean_text(text):
    # convert to lowercase
    text = text.lower()
    # remove special characters
    text = re.sub(r'[^0-9a-zA-Z]', ' ', text)
    # remove extra spaces
    text = re.sub(r'\s+', ' ', text)
    # remove stopwords
    text = " ".join(word for word in text.split() if word not in STOPWORDS)
    return text

In [6]:
# clean the messages
df['clean_text'] = df['messages'].apply(clean_text)
df.head()

,messages,label,clean_text
0,"Go until jurong point, crazy.. Available only ...",ham,go jurong point crazy available bugis n great ...
1,Ok lar... Joking wif u oni...,ham,ok lar joking wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,spam,free entry 2 wkly comp win fa cup final tkts 2...
3,U dun say so early hor... U c already then say...,ham,u dun say early hor u c already say
4,"Nah I don't think he goes to usf, he lives aro...",ham,nah think goes usf lives around though


## Input Split

In [7]:
X = df['clean_text']
y = df['label']

---
# Standardized ML Pipeline

**Step 1** — LazyPredict: automated baseline comparison of dozens of models  
**Step 2** — PyCaret: automated final pipeline (setup → compare → finalize)


In [ ]:
# ── Feature Vectorization (for ML pipeline) ──
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

_text_series = X
_target_series = y

# Drop NaN rows
import pandas as pd
_mask = _text_series.notna() & pd.Series(_target_series).notna()
_text_clean = _text_series[_mask].astype(str)
_target_clean = np.array(pd.Series(_target_series)[_mask])

_tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
_X_vectorized = _tfidf.fit_transform(_text_clean)
_y_vectorized = _target_clean

print(f'Vectorized shape: {_X_vectorized.shape}')
print(f'Target classes: {np.unique(_y_vectorized)}')


In [ ]:
# ── STEP 1: LazyPredict — Baseline Model Comparison ──
from sklearn.model_selection import train_test_split
from lazypredict.Supervised import LazyClassifier
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

_X = _X_vectorized.toarray() if hasattr(_X_vectorized, 'toarray') else np.array(_X_vectorized) if not isinstance(_X_vectorized, np.ndarray) else _X_vectorized
_y = np.array(_y_vectorized).ravel()

X_train, X_test, y_train, y_test = train_test_split(_X, _y, test_size=0.2, random_state=42)

clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
models, predictions = clf.fit(X_train, X_test, y_train, y_test)
print(models)

# ── Metrics Extraction ──
best_model_name = models.sort_values('Accuracy', ascending=False).index[0]
_best_row = models.loc[best_model_name]
lp_accuracy = float(_best_row.get('Accuracy', 0))
lp_f1 = float(_best_row.get('F1 Score', 0))
print(f'\n>>> Best LazyPredict model: {best_model_name}')
print(f'    Accuracy={lp_accuracy:.4f}  F1={lp_f1:.4f}')


In [ ]:
# ── STEP 2: PyCaret — Final Pipeline ──
from pycaret.classification import setup, compare_models, finalize_model, pull

# Prepare DataFrame (sample if needed for speed)
_max_rows, _max_cols = 5000, 2000
_X_sub = _X[:_max_rows, :min(_X.shape[1], _max_cols)]
_y_sub = _y[:_max_rows]

df_ml = pd.DataFrame(_X_sub, columns=[f'f{i}' for i in range(_X_sub.shape[1])])
df_ml['target'] = _y_sub

s = setup(data=df_ml, target='target', session_id=42, verbose=False)
best = compare_models(n_select=1)
pycaret_results = pull()
print(pycaret_results)

final_model = finalize_model(best)
pycaret_model_name = type(best).__name__

# Extract PyCaret metrics
_pc_best = pycaret_results.iloc[0]
pc_accuracy = float(_pc_best.get('Accuracy', 0))
pc_precision = float(_pc_best.get('Prec.', 0))
pc_recall = float(_pc_best.get('Recall', 0))
pc_f1 = float(_pc_best.get('F1', 0))

print(f'\nPyCaret Best: {pycaret_model_name}')
print(f'  Accuracy={pc_accuracy:.4f}  Precision={pc_precision:.4f}  Recall={pc_recall:.4f}  F1={pc_f1:.4f}')


---
## Model Governance — Persistence & Registry

Save trained model, feature vectorizer, metrics, and register in global project registry.


In [ ]:
import json, os
from datetime import datetime
from pathlib import Path
from joblib import dump

project_name = 'sms_spam_analysis'
_artifacts_dir = Path('..') / 'artifacts' / project_name
_artifacts_dir.mkdir(parents=True, exist_ok=True)

# Save trained model
dump(final_model, str(_artifacts_dir / 'model.joblib'))

# Save feature vectorizer
dump(_tfidf, str(_artifacts_dir / 'vectorizer.joblib'))

# Save metrics
_metrics = {
    'best_model_lazypredict': best_model_name,
    'pycaret_model': pycaret_model_name,
    'accuracy': pc_accuracy,
    'f1': pc_f1,
    'precision': pc_precision,
    'recall': pc_recall,
    'lp_accuracy': lp_accuracy,
    'lp_f1': lp_f1,
}
with open(str(_artifacts_dir / 'metrics.json'), 'w') as f:
    json.dump(_metrics, f, indent=2)

# Update global registry
_registry_path = Path('..') / 'artifacts' / 'global_registry.json'
_registry_path.parent.mkdir(parents=True, exist_ok=True)
if _registry_path.exists():
    with open(str(_registry_path)) as f:
        _registry = json.load(f)
else:
    _registry = []
_registry = [e for e in _registry if e.get('project') != project_name]
_registry.append({
    'project': project_name,
    'best_model': best_model_name,
    'pycaret_model': pycaret_model_name,
    'accuracy': pc_accuracy,
    'timestamp': datetime.now().isoformat(),
})
with open(str(_registry_path), 'w') as f:
    json.dump(_registry, f, indent=2)

print(f'Artifacts saved to {_artifacts_dir}/')


In [ ]:
# ── Inference Function ──
def predict_text(text):
    """Run inference on a single text input."""
    vec = _tfidf.transform([text])
    return final_model.predict(vec)

print('Inference function ready: predict_text(text)')


In [ ]:
# ── Consistency Checks ──
assert final_model is not None, 'Final model was not created'
assert best_model_name is not None, 'Best model name was not captured'
assert (_artifacts_dir / 'model.joblib').exists(), 'Model file not saved'
assert (_artifacts_dir / 'metrics.json').exists(), 'Metrics file not saved'

# ── Summary ──
print('=' * 50)
print('MODEL GOVERNANCE SUMMARY')
print('=' * 50)
print(f'Project:              {project_name}')
print(f'Best Model (LP):      {best_model_name}')
print(f'Best Model (PyCaret): {pycaret_model_name}')
print(f'Accuracy:             {pc_accuracy:.4f}')
print(f'Precision:            {pc_precision:.4f}')
print(f'Recall:               {pc_recall:.4f}')
print(f'F1 Score:             {pc_f1:.4f}')
print(f'Artifacts:            {_artifacts_dir}/')
print('=' * 50)
